# 2024-11-04 Removing atmospheric effects, round 1

## Overview

From last time, you have measuresments of the brightness of the stars in your color image.

Today we will talk about how to remove the effects of the atmosphere and our instruments on those measurements.

<!-- ![Sketch of atmospheric and intrumental effects](media/AST-266-27.jpg) -->

In [ ]:
from pathlib import Path
import numpy as np

from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table

from astroquery.gaia import Gaia

from stellarphot import PhotometryData

%matplotlib inline
import matplotlib.pyplot as plt

#### 👇👇 put in the name of your file here 👇👇

In [ ]:
your_mag_table = 'photometry.ecsv'

In [ ]:
mag_table = PhotometryData.read(your_mag_table)

In [ ]:
B = mag_table[mag_table['passband'] == 'B']
V = mag_table[mag_table['passband'] == 'V']

In [ ]:
mag_col = 'mag_inst'
# fig, resid = plt.subplots(1, 1, figsize=(10, 10))

# resid.plot(B['color_cat'], B['color_cat'] - (B[mag_col] - V[mag_col]), '.', label='catalog colors', alpha=0.4)
# #plt.ylim(16.5, 10)
# resid.set_xlabel('Catalog B-V')
# resid.set_ylabel('Diff between catalog and calibrated B-V')
# resid.set_title(your_mag_table)
# resid.grid()

In [ ]:
good_color = np.abs(B[mag_col]) > -1  # B['color_cat'] - (B['mag_inst_cal'] - V['mag_inst_cal'])) < 0.1

In [ ]:
fig, cmd = plt.subplots(1, 1, figsize=(5, 5))

cmd.plot(B[mag_col][good_color] - V[mag_col][good_color], V[mag_col][good_color], '.')
cmd.set_ylim(*cmd.get_ylim()[::-1])
cmd.set_ylabel('V')
cmd.set_xlabel('B - V')
cmd.set_xlim(-0.5, 2.5)
cmd.set_xlabel("Color (B-V instrumental)")
cmd.set_ylabel("Magnitude (V instrumental")
cmd.grid()


## Atmospheric and instrumental effects

Discussion on the ipad....

<!-- ![Sketch of atmospheric and intrumental effects](media/AST-266-26.jpg) -->

## Calibrating your magnitudes

In [ ]:
from pathlib import Path

import ipywidgets as ipw
import numpy as np

from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table, vstack

from astroquery.gaia import Gaia

from stellarphot import PhotometryData
from ipyautoui.custom import FileChooser

from astropy.coordinates import SkyCoord
from stellarphot import apass_dr9, PhotometryData
%matplotlib widget
from matplotlib import pyplot as plt

import numpy as np
from stellarphot.utils.magnitude_transforms import transform_to_catalog

In [ ]:
vb = ipw.VBox()
object_name = ipw.Text(description="M Object:")
file_label = ipw.HTML("Choose your photometry file")
chooser = FileChooser(filter_pattern=["*.ecsv"])
vb.children = [object_name, file_label, chooser]
vb

In [ ]:
coocoo = SkyCoord.from_name(object_name.value)

In [ ]:
your_mag_table = chooser.value

In [ ]:
pd = PhotometryData.read(chooser.value)
#pd = pd[pd["passband"] == "V"]

In [ ]:
pdv = pd[pd["passband"] == "V"]

In [ ]:
dr9 = apass_dr9(coocoo)
dr9_good = dr9.passband_columns(passbands=["B", "V", "SR"])
dr9_coord = SkyCoord(dr9_good['ra'], dr9_good['dec'], unit='degree')

In [ ]:
our_coord = SkyCoord(pdv['ra'], pdv['dec'])
idx, d2d, _ = our_coord.match_to_catalog_sky(dr9_coord)
good_matches = d2d.arcsec < 1

In [ ]:
fig, ax = plt.subplots()
x = pdv["mag_inst"][good_matches]
y = dr9_good[idx[good_matches]]["mag_V"].filled(np.nan)
b = dr9_good[idx[good_matches]]["mag_B"].filled(np.nan)
e = pdv["mag_error"][good_matches].value

not_bad  = np.isfinite(x) & np.isfinite(y) & np.isfinite(e)
x = x[not_bad]
y = y[not_bad]
e = e[not_bad]
b = b[not_bad]

ax.errorbar(
    x, 
    y,
    xerr=e,
    fmt=".",
    color="blue"
)
ax.set_ylabel("Catalog V magnitude")
ax.set_xlabel("Feder instrumental V magnitude")
ax.grid()

In [ ]:
a_delta = 0.5
b_min = -0.1
c_min = -0.5
d_min = -1e-6

our_filters = ["B", "V", "SR"]

aavso_band_names = dict(B="B", V="V", gp="SG", rp="SR", ip="SI", SI="SI", SR="SR")

# cat_color_colums = dict(
#     B=("Bmag", "Vmag"),
#     V=("Bmag", "Vmag"),
#     gp=("g_mag", "r_mag"),
#     rp=("r_mag", "i_mag"),
#     ip=("r_mag", "i_mag"),
#     SI=("r_mag", "i_mag"),
# )

cat_color_colums = dict(
    B=("B", "V"),
    V=("B", "V"),
    gp=("mag_SG", "mag_SR"),
    rp=("mag_SR", "mag_SI"),
    SR=("SR", "SI"),
    ip=("mag_SR", "mag_SI"),
    SI=("mag_SR", "mag_SI"),
)

# cat_filter = dict(
#     B="Bmag",
#     V="Vmag",
#     gp="g_mag",
#     rp="r_mag",
#     ip="i_mag",
#     SI="i_mag",
# )

cat_filter = dict(
    B="B",
    V="V",
    gp="mag_SG",
    rp="mag_SR",
    SR="SR",
    ip="mag_SI",
    SI="mag_SI",
)


In [ ]:
pd.add_bjd_col()
# # BAD BAD BAD BAD BAD BAD BAD BAD BAD BAD
# all_mags["passband"] = "SR"


# Ensure we have the right table ordering later
pd.sort(["passband", "bjd"])
filter_groups = pd.group_by("passband")

In [ ]:
output_table = []

for key, group in zip(filter_groups.groups.keys, filter_groups.groups):
    # The key is a table column, not a value...
    k = key[0]
    print(f"Transforming band {k}")
    by_bjd = group.group_by("file")

    transform_to_catalog(
        by_bjd,
        f"mag_inst",
        aavso_band_names[k],
        obs_error_column="mag_error",
        zero_point_range=[12, 25],
        c_delta=0.5,  # b_delta=0.1,
        cat_filter=cat_filter[k],
        cat_color=cat_color_colums[k],
        in_place=True,
    )
    output_table.append(by_bjd.copy())

output_table = vstack(output_table, join_type="outer")

In [ ]:
pd['mag_inst_cal'] = output_table['mag_inst_cal']
pd['color_cat'] = output_table['color_cat']

In [ ]:
from astroquery.gaia import Gaia

Gaia.ROW_LIMIT = -1
j = Gaia.cone_search_async(coocoo, radius=20 * u.arcmin)

In [ ]:
r = j.get_results()
gaia_cs = SkyCoord(ra=r['ra'], dec=r['dec'], unit='degree')

In [ ]:
our_coord_all = SkyCoord(pd['ra'], pd['dec'])
idxg, d2dg, _ = our_coord_all.match_to_catalog_sky(gaia_cs)

In [ ]:
great = d2dg < 1 * u.arcsec
distance = 1000 / r['parallax']

In [ ]:
abs_mag = 0 * pd['mag_inst_cal']
abs_mag[great] = pd['mag_inst_cal'][great] - 5 * (np.log10(distance[idxg[great]]) - 1)
pd['abs_mag'] = abs_mag

In [ ]:
def two_up(B, V, title, mag_col='mag_inst_cal'):
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    resid, cmd = ax
    good_color = np.abs(B['color_cat'] - (B['mag_inst_cal'] - V['mag_inst_cal'])) < 0.1

    #resid.plot(B['color_cat'], B['color_cat'] - (B[mag_col] - V[mag_col]), '.', label='catalog colors', alpha=0.4)
    #plt.ylim(16.5, 10)
    resid.plot(B["mag_inst_cal"][good_color] - V["mag_inst_cal"][good_color], V["mag_inst_cal"][good_color], '.')
    resid.set_ylim(*resid.get_ylim()[::-1])
    resid.set_xlim(-0.5, 2)
    resid.set_xlabel('B-V')
    resid.set_ylabel('Apparent V, calibrated')
    resid.set_title(title)
    resid.grid()
    good_color = np.abs(B['color_cat'] - (B['mag_inst_cal'] - V['mag_inst_cal'])) < 0.1
    
    cmd.plot(B[mag_col][good_color] - V[mag_col][good_color], V[mag_col][good_color], '.')
    cmd.set_ylim(*cmd.get_ylim()[::-1])
    cmd.set_ylabel('Absolute V, calibrated')
    cmd.set_xlabel('B - V')
    cmd.set_xlim(-0.5, 2)
    cmd.grid()
    

In [ ]:
B = pd[pd['passband'] == 'B']
V = pd[pd['passband'] == 'V']
good = (B['abs_mag'] != 0) & (V['abs_mag'] != 0) & ~np.isnan(B['abs_mag']) & ~np.isnan(V['abs_mag'])
two_up(B[good], V[good], "Yowza", mag_col='abs_mag')

## Compare to this: http://www.atlasoftheuniverse.com/hr.html